# RetinaNet results: anchor scales and frame sampling

We compare the saved experiment measurements using tables, validation examples and Plotly charts. Missing results remain **Not run** or **Partial**. Tables use the latest completed validation epoch, matching saved predictions. A partial run can have measurements without a completed five epoch result.

In [1]:
import sys
from pathlib import Path

import plotly.io
import polars as pl
from decouple import config
from IPython.display import Image, display

ROOT = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from hm2.experiment_names import EXPERIMENTS, experiment_label
from hm2.retinanet_analysis import summarize_batch

EXPERIMENT_LABELS = {name: experiment_label(name) for name in EXPERIMENTS}

batch_setting = config("RETINANET_BATCH_DIR", default="")
batches = sorted((ROOT / "runs" / "retinanet_overnight").glob("*/"))
batch_dir = Path(batch_setting) if batch_setting else (batches[-1] if batches else None)
summary = summarize_batch(batch_dir) if batch_dir is not None else None
(
    pl.from_dicts(summary["comparison"], infer_schema_length=None).with_columns(
        pl.col("experiment_id").replace_strict(EXPERIMENT_LABELS)
    )
    if summary
    else "Not run"
)

experiment_id,status,map,ap50,ap75,ap_small,ap_medium,ap_large,per_class_ap,epoch,global_step,image_exposures,train_seconds,train_frames,inference_seconds,fps,validation_protocol,ignore_policy,validation_images
str,str,null,null,null,null,null,null,struct[0],null,null,null,null,null,null,null,null,null,null
"""RetinaNet with standard anchors (every fifth frame)""","""Not run""",null,null,null,null,null,null,{},null,null,null,null,null,null,null,null,null,null
"""RetinaNet with smaller anchors (every fifth frame)""","""Not run""",null,null,null,null,null,null,{},null,null,null,null,null,null,null,null,null,null
"""RetinaNet with standard anchors (full training set)""","""Not run""",null,null,null,null,null,null,{},null,null,null,null,null,null,null,null,null,null
"""RetinaNet with smaller anchors (full training set)""","""Not run""",null,null,null,null,null,null,{},null,null,null,null,null,null,null,null,null,null


## Q1: Does anchor tuning change validation AP?

We compare smaller anchors against standard anchors separately for every-fifth-frame
and full-data training. We inspect mAP, AP50, AP75, size-specific AP, and class AP.
These are measured differences from one seed, not estimates of statistical significance.
We check completed_pair and epoch before interpreting a contrast. No conclusion is
available until the corresponding measurements exist.


In [2]:
if summary:
    display(
        pl.from_dicts(
            [{"Contrast": name, **values} for name, values in summary["differences"].items()],
            infer_schema_length=None,
        ).with_columns(
            pl.col("changed").replace_strict(EXPERIMENT_LABELS),
            pl.col("baseline").replace_strict(EXPERIMENT_LABELS),
        )
    )

Contrast,changed,baseline,completed_pair,map,ap50,ap75,ap_small,ap_medium,ap_large,train_seconds,inference_seconds,fps,per_class_ap
str,str,str,bool,null,null,null,null,null,null,null,null,null,struct[0]
"""anchors_stride""","""RetinaNet with smaller anchors (every fifth frame)""","""RetinaNet with standard anchors (every fifth frame)""",false,null,null,null,null,null,null,null,null,null,{}
"""anchors_full""","""RetinaNet with smaller anchors (full training set)""","""RetinaNet with standard anchors (full training set)""",false,null,null,null,null,null,null,null,null,null,{}
"""full_minus_stride_baseline""","""RetinaNet with standard anchors (full training set)""","""RetinaNet with standard anchors (every fifth frame)""",false,null,null,null,null,null,null,null,null,null,{}
"""full_minus_stride_tuned""","""RetinaNet with smaller anchors (full training set)""","""RetinaNet with smaller anchors (every fifth frame)""",false,null,null,null,null,null,null,null,null,null,{}


## Q2: What changes with the full training set?

We compare full-data training against every-fifth-frame training separately for standard
and smaller anchors. Five full-data epochs expose approximately five times as many frames
and optimizer updates as five stride-five epochs. This comparison confounds data coverage
and optimization budget. We inspect actual exposures, successful steps, elapsed training
time, validation AP, and inference speed before choosing a pipeline.


In [3]:
if summary:
    for chart in summary["charts"]:
        plotly.io.read_json(Path(chart).with_suffix(".json")).show(renderer="plotly_mimetype")

Plotly figure

Plotly figure

Plotly figure

## Consistent validation diagnostics

We use a consistent selection of a dense scene and tiny-object scene from validation metadata, plus a genuine missed-GT example when one exists in the first available run. It uses greedy class-aware IoU 0.5 matching at score ≥ 0.3. These are qualitative diagnostics, not the COCO evaluator. The same selected frames are shown for each model. Ignored regions remain visible in both panels.

In [4]:
dataset_setting = config("VISDRONE_ROOT", default="")
metadata_setting = config("VISDRONE_METADATA", default="")
if batch_dir is not None and dataset_setting and metadata_setting:
    summary = summarize_batch(batch_dir, Path(dataset_setting), Path(metadata_setting))
    for preview in summary["previews"]:
        display({**preview, "experiment_id": experiment_label(preview["experiment_id"])})
        display(Image(filename=preview["path"]))

## Evaluation boundaries and pending comparison

The evaluator uses original category-specific COCO crowd annotations, COCO IoU thresholds, and maxDets [1, 10, 100]. Crowd handling follows COCO semantics and does not establish equivalence to a detector-independent don't-care evaluator. The competition scoring semantics remain unverified; COCO AP must not be presented as Kaggle scoring. Preview filtering at score 0.3 does not change evaluation predictions.

The tuned Faster R-CNN comparison is **Not run** in this batch. We include its measured validation result under a matching protocol before selecting the final prediction pipeline. We reserve the final interpretation until the compared runs have completed under the same protocol.